# Project Checkpoint 2: RQ Formation

**Justin Liu – 830003581**  
**CSCE 676**

---

# (A) Project Scope

---

## Dataset and EDA Recap

Previously, exploratory analysis was performed on the SNAP Ego-Twitter Network graph, a directed follower graph that visualize the relationships between social cicles on Twitter. One of the conclusions drawn from the findings was that the network is large, structurally complexd, and sparse. The in-degree and out-degree distributions were also fairly skewed, showing the unequal user infleunce that is within the graph that is leaned towards less authoratative behavior in the space. While analyzing the SCCs and WCCs, we were able to show that the graph has meaningful connectivity structure. Furthermore, shortest path sampling wqas utilized in our analysis to show that the majority of users remained connected through short paths. Finally, community detection further suggested the presence of multiple smaller communities inside the entire network.

## Course Techniques to be Used/Already Used

  - Degree Analysis - simple baseline for influence
  - PageRank - captures global influence by considering neighboring nodes
  - HITS - important for capturing hub/authority roles
  - SCC/WCC Analysis - connectivity structure baseline

## External Techniques to be Used/Already Used

  - Node2Vec
  

---

# (B) Research Questions

---

## Research Question 1: How do Node Embeddings Reveal Similarity Patterns not Visible through Tradtional Graph Metrics?

  - Task Type: Representation Learning / Similarity Mining
  - Relevant Algorithms: Node2Vec
  - Evaluation Criteria: Nearest-Neighbor Quality, Interpretability of Embedding Neighborhoods

## Research Question 2: How does the Community Structure Relate to the Influence and Connectivity in the Graph?
  
  - Task Type: Community Mining / Structural Graph Analysis
  - Relevant Algorithms: PageRank, HITS
  - Evaluation Criteria: Modularity, Average Centrality

## How do different Graph-Based Influence Measures compare in identifying more Important Users in the Twitter Network?

  - Task Type: Graph Ranking
  - Relevant Algorithms: In-Degree, Out-Degree, PageRank, HITS
  - Evaluation Criteria: Top-k Overlap, Rank Correlation


---

# (C) Research Question - to - Method Mapping Table

---

Below is a mapping table summarizing the research questions above:

| Research Question | Data Mining Task Type  | Course or External | Algorithms | Evaluation Criteria |
|---|---|---|---|---|
| Embedding-based Similarity  | Representation Learning / Similarity Mining  | External  | Node2Vec  | Nearest-Neighbor Quality, Interpretability of Embedding Neighbors  |
| Community V.S Influence  | Community Mining / Structural Graph Analysis  | Course  | PageRank, HITS  | Modularity, Average Centrality  |
| Influence Comparison  | Graph Ranking  | Course  | In-Degree, Out-Degree, PageRank, HITS  | Top-k Overlap, Rank Correlation  |




---

# (D) Exploratory Data Analysis for Research Questions

---



In [15]:
# ------------------------------------------------------
# 1. Download Directed Graph
# ------------------------------------------------------
# Source Page: https://snap.stanford.edu/data/ego-Twitter.html
# File name on SNAP: https://snap.stanford.edu/data/twitter_combined.txt.gz

import os, io, gzip, zipfile, tarfile, sys, math, random
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

# download dataset
DATA_URL = "https://snap.stanford.edu/data/twitter_combined.txt.gz"
LOCAL_PATH = "data/twitter_combined.txt.gz"

os.makedirs("data", exist_ok=True)

def download_dataset(url: str, to_path: str):
    import urllib.request
    print(f"Downloading from {url} ...")
    urllib.request.urlretrieve(url, to_path)
    size = os.path.getsize(to_path) / (1024*1024)
    print(f"Saved to {to_path} ({size:.2f} MB)")

download_dataset(DATA_URL, LOCAL_PATH)

Saved to data/twitter_combined.txt.gz (10.13 MB)


## Downloading Data

The ego-Twitter network dataset is obtained from the SNAP repository owned by Stanford. This directed grpah represents follower relationships between users.

In [16]:
# ------------------------------------------------------
# 2. Load Directed Graph
# ------------------------------------------------------
import gzip

def load_directed_graph(path, comment='#', sep=None) -> nx.DiGraph:
  '''
  Load directed edge list into a DiGraph
  Assumes there are comment lines
  Assumes lines are formatted like: u<sep>v
  '''
  graph = nx.DiGraph()

  # line counters
  line_count = 0
  skipped_lines_count = 0

  open_fn = gzip.open if path.endswith('.gz') else open

  with open_fn(path, 'rt') as f:
    for line in f:
      line_count += 1

      # ignore lines that start with # (comments)
      if line.startswith(comment):
        skipped_lines_count += 1
        continue

      components = line.strip().split(sep)

      # ignore lines that are malformed
      if len(components) != 2:
        skipped_lines_count += 1
        continue

      try:
        src, dst = map(int, components)
        graph.add_edge(src, dst)
      except ValueError:
        skipped_lines_count += 1
        continue

    return graph

print('Loading graph...')
graph = load_directed_graph(LOCAL_PATH)
print('Graph loaded.')


Loading graph...
Graph loaded.


## Loading Graph

The dataset is parsed into a NetworkX DiGraph which will be utilized for exploratory data analysis. As we parse through the data, we make sure to clean the data - remove comment lines or malformed entries. Edge endpoints ar econverted to integer form for efficiency and consistency purposes.

In [17]:
# ------------------------------------------------------
# 3. Degree V.S PageRank Overlap
# ------------------------------------------------------
pagerank = nx.pagerank(graph)

# dataframe
df1 = pd.DataFrame({'node': list(graph.nodes()), 'in_degree': [degree for _, degree in graph.in_degree()]})

df1['pagerank'] = df1['node'].map(pagerank)

# top-k overlap
k = 10
top_degree  = set(df1.sort_values('in_degree', ascending=False).head(k)['node'])
top_pagerank  = set(df1.sort_values('pagerank', ascending=False).head(k)['node'])

overlap = len(top_degree & top_pagerank)
print(f'Top-{k} overlap (Degree V.S PageRank): {overlap}/{k}')

Top-10 overlap (Degree V.S PageRank): 5/10


## Degree V.S PageRank Overlap

The partial overlap ebtween degree-based and RageRank based rankings suggest that the local connectivity itself does not determine the entire influence of the graph. This implies that different influence measures can produce different perspectives of which users are the most influential.

In [18]:
# ------------------------------------------------------
# 4. SCC Structure
# ------------------------------------------------------

# compute sccs
sccs = list(nx.strongly_connected_components(graph))
scc_sizes = [len(c) for c in sccs]

# largest scc information
largest_scc = max(sccs, key=len)
G_scc = graph.subgraph(largest_scc).copy()
print(f'Largest SCC nodes: {G_scc.number_of_nodes()}')
print(f'Largest SCC edges: {G_scc.number_of_edges()}')

# utilize pagerank from rq1
df2 = pd.DataFrame({'node': list(graph.nodes()), 'pagerank': [pagerank[n] for n in graph.nodes()], 'in_degree': [d for _, d in graph.in_degree()]})
df2['in_scc'] = df2['node'].apply(lambda x: x in largest_scc)

# compare
scc_statistics = df2.groupby('in_scc').agg({'pagerank': 'mean', 'in_degree': 'mean', 'node': 'count'}).rename(columns={'node': 'size'})
display(scc_statistics)

Largest SCC nodes: 68413
Largest SCC edges: 1685163


,pagerank,in_degree,size
in_scc,,,
False,0.000007,6.429070,12893
True,0.000013,24.633608,68413


## SCC Comparison

The SCC comparison shows that the nodes inslide the largest SCC tend to have higher average PageRank and in-degree than the nodes outside of it. This may lead to the assumption that influence is more concentrated in the graph's core.

---

# (E) Motivation and Feasibility

---

## Motivation

Building off of Checkpoint 1, the Twitter follower network exhibits several structural properities that intrigue me enough to perform deeper graphing mining analysis. Based on prior EDA, the graph is large, sparse, and highly skwed in degree distribution. These suggests that there are a good amount of influential users, but now the question is how should that influence be defined and measured.

The additional analysis I conducted briefly dive into the second and third search question. There is partial agreement on the rankings between degree-based and PageRank, indicating that the local connectivity does not fully capture the global infleunce. Furthermore, while earlier analysis showed the graph's strong overall connectivity, the SCC analysis provides a more subtle view of the network structure. Nodes within the largest SCC exhibit higher average RageRank and in-degree values compared to the nodes outside of it, which signifies a big concentration within the central core.

Finally, as previously observed in Checkpoint 1, the graph's sparse nature indicates the assymetric relationships in the graph. These relationships lead me to believe that tradtional graph metrics may not capture node similarity as well compared to other methods like NOde2Vec, where latent representations are learned to capture higher-order structural relationships.

## Non-triviality

Diferent data mining techniques capture different aspects of the network. For example, degree centrality reflects local connectivity and PageRank captures global influence through relationships. SCC analysis reveals that structural roles are not clear from just analyzing degrees alone and Node2Vec captures higher-order strucutral similarity beyond established, direct connections.

These specific methods operate at different levels, and it is not guaranteed that having one specific method can address a lot of the questions proposed. This complexity makes the analysis both non-trivial and meaningful.

## Feasibility

The methods proposed are computationall feasible based on prior and additional analysis. The graph is manageable enoguh in size and suitable for analysis utilizing graph processing tools. Additonal analysis confirms that the runtime of specific methods such as PageRank and HITS can be applied directly without any additional or preprocessing procedures.

Runtime experiments support feasibility. As seen from the runtimes of both HITS and PageRank, it can be safe to assume that other centrality-based methods can be executed just as efficieintly on the full graph. These results indicate that the core algorithms necessary to answer the research questions are practical to run.

## Risks

Computational costs and parameter selection are primary risks for this project, especially for Node2Vec. Embeedding quality may depend on hyperameters, and poor choices may lead to failure to capture meaningful structural relationships.

Another potential challenge would be the interpretability of embedding results. However, this can be addressed by analyzing nearest neighbors and comparing embedding similarity to other traditional graphing metrics.

---

# (F) Exploratory Data Analysis to Test Feasibility

---

In [19]:
# ------------------------------------------------------
# 3. Largest WCC Coverage
# ------------------------------------------------------

wccs = list(nx.weakly_connected_components(graph))
largest_wcc = max(wccs, key=len)

coverage = len(largest_wcc) / graph.number_of_nodes()

print(f'Largest WCC Size: {len(largest_wcc):,}')
print(f'Coverage: {coverage:.2%}')

Largest WCC Size: 81,306
Coverage: 100.00%


## Largest WCC Coverage

Our entire graph is one connected component. This is great because this means that we do not need to discard nodes and global methods such as Node2Vec should be fully applicable.

In [20]:
# ------------------------------------------------------
# 4. PageRank Runtime Feasibility Check
# ------------------------------------------------------

import time

start_pagerank = time.time()
pagerank = nx.pagerank(graph)
end_pagerank = time.time()

print(f'PageRank Runtime (WCC): {end_pagerank - start_pagerank:.2f} seconds')

PageRank Runtime (WCC): 2.24 seconds


## PageRank Runtime Feasibility

If PageRank can be computed within a reasonable runtime on the largest connected component, then other centrality-based methods such as HITS should be computationally feasible as well

In [21]:
# ------------------------------------------------------
# 5. PageRank Runtime Feasibility Check
# ------------------------------------------------------

start_hits = time.time()
hubs, authorities = nx.hits(graph, max_iter=1000, normalized=True)
end_hits = time.time()

print(f'PageRank Runtime (WCC): {end_hits - start_hits:.2f} seconds')

PageRank Runtime (WCC): 2.28 seconds


## HITS Runtime Feasibility Check

The runtime of HITS and PageRank are extremely similar and comparable, further validating that other centrality-based methods can be applied efficiently on this graph.

---

# (F) Methodological Planning

---

## Course Algorithms

For this project, here are some of the techniques covered in the course that I plan to use to analyze influence and structural properties of the network.

### PageRank

  - PageRank can be utilized to capture global influence based of analyzing neighboring nodes.

### HITS

  - In a graph where follower activity is prevalent, it is important to be able to distinguish between hub-like and authority-like behavior.

### Degree Centrality
  
  - Degree measures are going to be utilzied as a baseline for influence. This is perfect for analyzing how many followers a node has and reflecting activity in following others

## External Algorithms

### Node2Vec

  - For this project, Node2Vec will be utilized as an external algorithm to learn node embeddings. With this method, we hope to capture higher-order structural relationships and similarities between nodes. This opens up for more similarity analysis that goes beyond just tradtional centrality measures.

## Evaluation

For this project, the evaluation will be desigend to access how different methods capture meaningful pattenrs of influence, structure, and similarity within the network. For question 1, the focus will be on whether embedding-based methods can capture meaningful relationships that are not visible from traditional graphing methods. In the future, the aim is to be able to examine similarity patterns among nodes and access whether representations can align with already known characteristics of the network. From this evaluation, we hope to see whether the embeddings reveal interesting insights compared to other methods.

Furthermore, the second research question's evaluation will focus on how the structural properities of the graph relate to infleunce. By conducting a further analysis on the structure - analyzing differences betrween speciifc nodes, we hope to see how influence is dispersed across the entire graph and how that affects how the entire social graph works.

Finally, the third research question's evaluation will focus on comaparing all these metrics together to determine whether the influence compared is all consistent or diffeering. By examining how the rankings vary across different methods and metrics, we can further analyze the various roles of specific nodes in the graph.

## Baselines

For this project, simple-baseline methods are included. This means baselines such as degree-based ranking, random node comparing, and structural metrics will serve as a guideline to evaluate and provide insight into answering the research questions.

# Collaboration Declaration

---

## Collaborators:
  - Project was done independently without the contribution of other students

## Web Sources:
  - Jure Leskovec and Andrej Krevl. SNAP Datasets: Stanford Large Network Dataset Collection. http://snap.stanford.edu/data
  - NetworkX Documentation: https://networkx.org/documentation/stable/reference/index.html

## AI Tools:
I utilized ChatGPT to help me with:
  - Organize the notebook,
  - Strengthen methodological explanations
  - Revise explanations for clarity
  - Prompt more EDA ideas different from previos checkpoint
  
## Citations for Papers:
  - None
